# The k-sweep — does context size explain the oracle gap?

**The observation this tests:**

| Condition | Passages | Gold present | Correctness |
|---|---|---|---|
| C5 (reranked, top-5) | 5 | 96.0% | 84.5% |
| C4 (oracle) | 1 | 100% | 96.0% |

The 11.5-point correctness gap is far larger than the 4-point gold-presence gap. Something about the 4 extra (mostly wrong) passages is costing more than the small remaining chance the gold is missing.

**Why this is testable now, and wasn't before:** pre-reranking, Recall@1 was only 0.620, so k=1 would have dropped the gold passage 38% of the time — too reckless to try. Post-reranking it's 0.800, making k=1–3 a real option.

**Runs the full pipeline at k ∈ {1, 2, 3, 5}**, holding everything else fixed. Retrieval-level cost (Recall@k falling) is separated from generation-level effect (does correctness rise as distractors are removed?).

**Both outcomes are informative** — confirms distractors matter (→ "rerank AND shrink k"), or rules that hypothesis out (→ the oracle gap is something else, itself worth reporting).

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **GPU required.**

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers transformers accelerate bitsandbytes

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "base_encoder": "intfloat/multilingual-e5-base",
    "reranker": "BAAI/bge-reranker-v2-m3",
    "alpha": 0.8,
    "retrieve_k": 20,          # candidates fed to the reranker (unchanged from solution_6)
    "k_sweep": [1, 2, 3, 5],   # passages kept AFTER reranking, for the generator

    "n_eval": 200,
    "llm": "Qwen/Qwen2.5-7B-Instruct",
    "load_4bit": True,
    "max_new_tokens": 128,
    "batch_size": 8,

    "checkpoint": "k_sweep_checkpoint.json",
    "seed": 42,
}

### Load data

In [ ]:
import json, random, re, os, gc
import numpy as np
import pandas as pd

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]

# Same seed as solution_6, so this run's C5(k=5) should reproduce that run's C5.
random.Random(CONFIG["seed"]).shuffle(qa)
eval_qa = qa[: CONFIG["n_eval"]]
print(f"Corpus {len(corpus)} | evaluating {len(eval_qa)} (same seed as solution_6)")

### BM25

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Retrieve + rerank ONCE at max depth; every k is a prefix of this

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

print("Building retrieval index...")
bi = SentenceTransformer(CONFIG["base_encoder"])
corpus_emb = np.asarray(
    bi.encode([f"passage: {t}" for t in corpus_texts],
              normalize_embeddings=True, batch_size=32, show_progress_bar=True), "float32")

def retrieve(query, k):
    q = bi.encode([f"query: {query}"], normalize_embeddings=True)[0]
    s = (CONFIG["alpha"] * minmax(corpus_emb @ q)
         + (1 - CONFIG["alpha"]) * minmax(np.asarray(bm25.get_scores(tokenize(query)))))
    return [corpus_ids[i] for i in np.argsort(-s)[:k]]

RK = CONFIG["retrieve_k"]
raw_candidates = {q["id"]: retrieve(q["darija_query"], RK) for q in eval_qa}
print(f"Retrieved top-{RK} for all {len(eval_qa)} Darija queries.")

del bi, corpus_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\nReranking with {CONFIG['reranker']}...")
from sentence_transformers import CrossEncoder
ce = CrossEncoder(CONFIG["reranker"], max_length=512, trust_remote_code=True,
                  automodel_args={"torch_dtype": torch.float32})

reranked_full = {}
for q in eval_qa:
    cands = raw_candidates[q["id"]]
    pairs = [(q["darija_query"], corpus_map[c]) for c in cands]
    scores = np.asarray(ce.predict(pairs, batch_size=16, show_progress_bar=False))
    if scores.ndim > 1:
        scores = scores[:, -1]
    order = np.argsort(-scores)
    reranked_full[q["id"]] = [cands[i] for i in order]

del ce
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Reranking complete -- every k below is a prefix of this ranking.")

### Retrieval-level cost of shrinking k (Recall@k, by definition)

In [ ]:
print("\n" + "=" * 78)
print("RETRIEVAL-LEVEL COST OF SHRINKING k (Darija, reranked)")
print("=" * 78)
for k in CONFIG["k_sweep"]:
    hit = np.mean([q["source_chunk_id"] in reranked_full[q["id"]][:k] for q in eval_qa])
    print(f"  k={k:<2} gold in context: {hit:.3f}")
print("\nThis is the retrieval-side price of shrinking context -- expected to fall")
print("monotonically. The question is whether generation-side correctness rises")
print("enough to be worth it.")

### Load the LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
) if CONFIG["load_4bit"] else None

tok = AutoTokenizer.from_pretrained(CONFIG["llm"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

llm = AutoModelForCausalLM.from_pretrained(
    CONFIG["llm"], quantization_config=quant, device_map="auto", torch_dtype=torch.float16)
llm.eval()
print(f"Loaded {CONFIG['llm']}")

@torch.no_grad()
def chat_batch(prompts, max_new_tokens=None):
    mnt = max_new_tokens or CONFIG["max_new_tokens"]
    texts = [tok.apply_chat_template([{"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True) for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=3072).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=mnt, do_sample=False, pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tok.decode(g, skip_special_tokens=True).strip() for g in gen]

print("Smoke test:", chat_batch(["أجب بكلمة واحدة: ما عاصمة المغرب؟"], 20)[0])

### Prompts (Arabic-only, same as solution_6)

In [ ]:
GEN_PROMPT = """أجب عن السؤال التالي اعتمادا فقط على النصوص المرفقة.

قواعد إلزامية:
- أجب بالعربية فقط. ممنوع استعمال أي كلمة بحرف لاتيني.
- إذا لم تكن الإجابة موجودة في النصوص، اكتب بالضبط: المعلومة غير متوفرة في النصوص
- لا تستعمل أي معرفة خارجية.
- أجب بجملة واحدة قصيرة فقط.

النصوص:
{context}

السؤال: {question}

الإجابة:"""

JUDGE_PROMPT = """النصوص المرجعية:
{context}

السؤال: {question}
الإجابة الصحيحة: {gold}
الإجابة المقدمة: {answer}

أجب عن سؤالين بدقة:
1. هل كل ما ورد في الإجابة المقدمة مدعوم صراحة بالنصوص المرجعية؟
2. هل الإجابة المقدمة مطابقة في المعنى للإجابة الصحيحة؟ اختلاف الصياغة مقبول، أما اختلاف الأرقام أو الأسماء أو التواريخ فغير مقبول.

أجب بهذا الشكل فقط وبدون أي شرح:
مدعوم: نعم/لا
مطابق: نعم/لا"""

def ctx_text(chunk_ids):
    return "\n\n".join(f"[{i+1}] {corpus_map[c]}" for i, c in enumerate(chunk_ids))

def parse_judge(text):
    t = (text or "").replace("،", " ")
    faithful = correct = 0
    for line in t.split("\n"):
        if "مدعوم" in line:
            faithful = 1 if "نعم" in line else 0
        elif "مطابق" in line:
            correct = 1 if "نعم" in line else 0
    return faithful, correct

### Run generation + judging at every k (batched, checkpointed)

In [ ]:
from tqdm.auto import tqdm

records = []
if os.path.exists(CONFIG["checkpoint"]):
    records = json.load(open(CONFIG["checkpoint"], encoding="utf-8"))
    print(f"Resuming with {len(records)} records.")
done = {(r["qid"], r["k"]) for r in records}
B = CONFIG["batch_size"]

for k in CONFIG["k_sweep"]:
    todo = [q for q in eval_qa if (q["id"], k) not in done]
    if not todo:
        print(f"\n=== k={k} === (cached, skipping)")
        continue
    print(f"\n=== k={k} ({len(todo)} to do) ===")
    for i in tqdm(range(0, len(todo), B)):
        batch = todo[i:i + B]
        chunks = [reranked_full[q["id"]][:k] for q in batch]
        answers = chat_batch([GEN_PROMPT.format(context=ctx_text(c), question=q["darija_query"])
                              for q, c in zip(batch, chunks)])
        verdicts = chat_batch([JUDGE_PROMPT.format(context=ctx_text(c), question=q["msa_query"],
                                                   gold=q["gold_answer"], answer=a)
                               for q, c, a in zip(batch, chunks, answers)], 40)
        for q, c, a, v in zip(batch, chunks, answers, verdicts):
            f, ok = parse_judge(v)
            records.append({
                "qid": q["id"], "k": k,
                "gold_in_context": int(q["source_chunk_id"] in c),
                "n_passages": len(c),
                "answer": a, "faithful": f, "correct": ok,
                "refused": int("غير متوفرة" in (a or "")),
            })
        json.dump(records, open(CONFIG["checkpoint"], "w", encoding="utf-8"), ensure_ascii=False, indent=2)

gen = pd.DataFrame(records)
gen.to_csv("k_sweep_raw.csv", index=False)
print(f"\nComplete: {len(gen)} generations across {gen.k.nunique()} values of k.")

### The sweep: does correctness rise as k shrinks?

In [ ]:
print("=" * 78)
print("K-SWEEP RESULTS")
print("=" * 78)
summary = gen.groupby("k").agg(
    n=("qid", "count"), gold_in_context=("gold_in_context", "mean"),
    faithfulness=("faithful", "mean"), correctness=("correct", "mean"),
    refusal_rate=("refused", "mean"),
).sort_index()
print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
summary.to_csv("k_sweep_summary.csv")

### Statistical test: k=1 vs k=5, paired

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])

def paired(a_k, b_k, col):
    a = gen[gen.k == a_k].set_index("qid")[col]
    b = gen[gen.k == b_k].set_index("qid")[col]
    common = a.index.intersection(b.index)
    d = (a.loc[common] - b.loc[common]).values.astype(float)
    idx = rng.integers(0, len(d), size=(1000, len(d)))
    m = d[idx].mean(axis=1)
    lo, hi = np.percentile(m, [2.5, 97.5])
    return d.mean(), lo, hi

print("\n" + "=" * 78)
print("DOES SHRINKING k IMPROVE CORRECTNESS? (paired 95% CI)")
print("=" * 78)
ks = sorted(gen.k.unique())
rows = []
for k in ks:
    if k == max(ks):
        continue
    d, lo, hi = paired(k, max(ks), "correct")
    sig = "yes -- fewer distractors help" if lo > 0 else \
          ("yes -- fewer distractors HURT" if hi < 0 else "no")
    print(f"  k={k} vs k={max(ks)}   correctness diff {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  {sig}")
    rows.append({"k": k, "vs_k": max(ks), "diff": d, "lo": lo, "hi": hi, "verdict": sig})
pd.DataFrame(rows).to_csv("k_sweep_comparisons.csv", index=False)

### Verdict against the oracle hypothesis

In [ ]:
print("\n" + "=" * 78)
print("VERDICT")
print("=" * 78)
best_k = summary["correctness"].idxmax()
worst_k = summary["correctness"].idxmin()
print(f"""
Best correctness:  k={best_k}  ({summary.loc[best_k, 'correctness']:.3f})
Worst correctness: k={worst_k}  ({summary.loc[worst_k, 'correctness']:.3f})

Recall the oracle (1 passage, always correct) scored 0.960 correctness.
k=1 here uses 1 passage but is NOT always correct (Recall@1 < 1.0), so it
isolates the distractor-count effect from the always-correct-context effect.

If correctness rises monotonically as k shrinks despite gold_in_context also
falling, distractors are confirmed as a real cost independent of retrieval
accuracy -- the paper's recommendation becomes "rerank AND use a small k",
not "rerank alone with a large k for safety."

If correctness does NOT rise as k shrinks, the oracle's advantage is better
explained by something else (e.g. the generator behaving differently with
exactly one passage vs. a multi-passage list), and this sweep is itself a
useful negative result ruling out the distractor-count hypothesis.
""")

from google.colab import files
files.download("k_sweep_raw.csv")
files.download("k_sweep_summary.csv")
files.download("k_sweep_comparisons.csv")